# LED OCR ablation, phase 2: extract pipeline inputs (Colab T4)
Protocol `led-ocr-ablation-261110-v1`. Run the SMOKE set first, read the timings, then run COHORT.
Videos and outputs stay on a private Drive folder, never in the repo. A failed video is a ledger row, not a rerun.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, subprocess
from pathlib import Path

APP = '/content/multimodal-lecture-summarizer'   # app repo (separate repo)
BENCH = '/content/benchmark'                     # benchmark repo (separate repo, private)
WORK = '/content/drive/MyDrive/led-ocr-ablation-261110'   # private: videos/, run/
SET = 'smoke'                                    # 'smoke' first, then 'cohort'
APP_COMMIT = '5e26ea799de9ac7e5a934d684ed4a031ff1202e4'  # frozen in protocol.json
PY = '/content/venv/bin/python'                  # same layout as the app's Colab worker notebook
PROTOCOL = f'{BENCH}/experiments/registry/protocols/led-ocr-ablation-261110-v1'

# benchmark repo is private: add a Colab secret GITHUB_TOKEN (read access) and enable notebook access.
try:
    from google.colab import userdata
    _tok = userdata.get('GITHUB_TOKEN')
except Exception:
    _tok = ''

def _clone(repo, dest):
    if os.path.isdir(dest): return
    base = f'x-access-token:{_tok}@' if _tok else ''
    r = subprocess.run(['git', 'clone', '-q', f'https://{base}github.com/multimodal-lecture-summarizer/{repo}.git', dest],
                       env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
    assert r.returncode == 0, f'clone of {repo} failed (private repo? set the GITHUB_TOKEN Colab secret)'

_clone('benchmark', BENCH); _clone('multimodal-lecture-summarizer', APP)
subprocess.run(['git', '-C', APP, 'checkout', '-q', APP_COMMIT], check=True)  # the scripts refuse any other app commit
os.makedirs(f'{WORK}/videos', exist_ok=True); os.makedirs(f'{WORK}/run', exist_ok=True)

hashes = json.load(open(f'{PROTOCOL}/hashes.json'))['sha256']
for name, expected in hashes.items():
    assert hashlib.sha256(open(f'{PROTOCOL}/{name}', 'rb').read()).hexdigest() == expected, name
print('frozen files verified')

In [ ]:
# Python 3.11 venv as in the app's Colab worker; Colab's own Python has no torch 2.5.1 wheels.
if not Path(PY).is_file():
    subprocess.run(['uv', 'venv', '--python', '3.11', '/content/venv'], check=True)
subprocess.run(['uv', 'pip', 'install', '--python', PY, '-r', f'{APP}/ai_workers/requirements.txt',
                '-r', f'{APP}/backend/requirements.txt', 'setuptools'], check=True, cwd=APP)
subprocess.run(['uv', 'pip', 'install', '--python', PY, '--no-deps', '-r', f'{APP}/ai_workers/requirements-tracking.txt'], check=True, cwd=APP)
subprocess.run(['nvidia-smi', '-L'])

In [ ]:
site = '/content/venv/lib/python3.11/site-packages'
ENV = {**os.environ,
       'LD_LIBRARY_PATH': f'{site}/nvidia/cudnn/lib:{site}/nvidia/cublas/lib:' + os.environ.get('LD_LIBRARY_PATH', ''),
       'PATH': '/content/venv/bin:' + os.environ['PATH'],
       'PADDLEOCR_USE_GPU': '0', 'CUBLAS_WORKSPACE_CONFIG': ':4096:8',
       'PROCESS_MIN_AVAILABLE_MEMORY_MB': '4096', 'PROCESS_SOFT_MIN_AVAILABLE_MEMORY_MB': '3072',
       'HF_HOME': '/content/hf'}

def run(cmd, cwd=None):
    """Stream output to the cell and stop on a non-zero exit."""
    p = subprocess.Popen(cmd, cwd=cwd, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout: print(line, end='')
    assert p.wait() == 0, f'step failed with exit code {p.returncode}; read the output above'

In [ ]:
# Resumable: rerun this cell after a disconnect. Videos with a ledger row are skipped.
run([PY, '-m', 'experiments.scripts.led_ocr_ablation_extract', '--app-root', APP, '--videos-dir', f'{WORK}/videos',
     '--out-dir', f'{WORK}/run', '--set', SET, '--protocol-dir', PROTOCOL], cwd=BENCH)

In [ ]:
# Descriptive tables only: by stratum, OCR volume per video, segmenter fallbacks. Not used to drop videos.
import pandas as pd
rows = [json.loads(l) for l in open(f'{WORK}/run/extract_ledger_{SET}.jsonl')]
df = pd.DataFrame(rows)
cols = [c for c in ['id','stratum','status','segmenter','scenes','keyframes_with_ocr','raw_ocr_words','merge_ocr_words','merge_slides_used','merge_transcript_words','asr_model','wall_seconds','error','detail'] if c in df]
display(df[cols])
if 'segmenter' in df: print('segmenter counts:', df['segmenter'].value_counts(dropna=False).to_dict())